# 02 — Regime Detection

**Purpose:** Upgrade the binary 200-day SMA trend filter in `rules_engine.py` to a probabilistic, composite stress regime model. Combines realised volatility and cross-asset correlation compression into a single stress score (0–1) and regime label per day.

**Why this matters:** The current rules engine moves to cash when an asset falls below its 200-day MA. This is binary and reactive — it triggers *after* a regime shift, not *as* it happens. A probabilistic model gives continuous early warning.

**Run time:** ~1–2 minutes.

**Output:** `research/outputs/regime_state.json`

---
**Sections:**
1. Setup & data fetch
2. Realised volatility regime
3. Correlation compression
4. Composite stress score
5. Regime probabilities
6. Transition statistics
7. Visualisations
8. Comparison: new model vs existing 200-day SMA filter
9. Export to JSON

## 1. Setup & data fetch

In [ ]:
import sys
import os
import json
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Path setup ──
NOTEBOOK_DIR   = os.path.abspath('')
RESEARCH_ROOT  = os.path.join(NOTEBOOK_DIR, '..')
PORTFOLIO_ROOT = os.path.join(RESEARCH_ROOT, '..', 'portfolio')

sys.path.insert(0, PORTFOLIO_ROOT)
sys.path.insert(0, RESEARCH_ROOT)

# ── Research imports ──
from src.config import (
    ASSET_UNIVERSE, BENCHMARK_TICKER, LOOKBACK_DAYS, RISK_FREE_RATE,
    REGIME_VOL_LOOKBACK, REGIME_VOL_LOW, REGIME_VOL_HIGH,
    REGIME_CORR_COMPRESSION_HIGH, REGIME_PROB_LOOKBACK, OUTPUT_REGIME,
)
from src.regime import (
    compute_realised_vol,
    classify_vol_regime,
    compute_correlation_compression,
    compute_composite_regime,
    compute_regime_probabilities,
    compute_transition_stats,
)

# ── Portfolio data layer ──
from src.data_loader import (
    fetch_historical,
    calculate_log_returns,
    fetch_fx_rate,
    convert_usd_prices_to_eur,
    load_ledger,
)
from src.math_optimizer import run_all_scenarios

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        11,
})

print('✅ Imports OK')

In [ ]:
# ── Fetch & prepare data ──

CACHE_PATH  = os.path.join(PORTFOLIO_ROOT, 'data', 'historical_prices.csv')
LEDGER_PATH = os.path.join(PORTFOLIO_ROOT, 'data', 'ledger.csv')

prices_raw = fetch_historical(ASSET_UNIVERSE, LOOKBACK_DAYS, CACHE_PATH)
usd_eur    = fetch_fx_rate('USD', 'EUR')
prices     = convert_usd_prices_to_eur(prices_raw, usd_eur)
log_returns = calculate_log_returns(prices)

# Portfolio weights from the optimizer (approximation of current weights)
holdings, cash = load_ledger(LEDGER_PATH)
held_tickers   = [t for t in holdings.keys() if t in log_returns.columns]

# Build an equal-weighted portfolio return series from held tickers
# (or use the optimizer weights if holdings exist)
if held_tickers:
    latest_prices = prices.iloc[-1]
    held_values   = {t: holdings[t] * float(latest_prices[t]) for t in held_tickers}
    total_held    = sum(held_values.values()) + cash
    held_weights  = {t: v / total_held for t, v in held_values.items()} if total_held > 0 else {}

    portfolio_returns = pd.Series(0.0, index=log_returns.index)
    for t, w in held_weights.items():
        portfolio_returns += log_returns[t] * w
    print(f'Portfolio returns: built from {len(held_tickers)} held tickers')
else:
    # Fallback: use benchmark
    portfolio_returns = log_returns[BENCHMARK_TICKER]
    print(f'⚠️  No held tickers found — using benchmark ({BENCHMARK_TICKER}) as portfolio proxy')

# Also keep benchmark returns for comparison
benchmark_returns = log_returns[BENCHMARK_TICKER]

print(f'\n✅ Data ready')
print(f'   Log returns shape : {log_returns.shape}')
print(f'   Portfolio series  : {len(portfolio_returns)} observations')
print(f'   Date range        : {log_returns.index[0].date()} → {log_returns.index[-1].date()}')

## 2. Realised volatility regime

First component of the composite model. Classifies each day as low / medium / high stress
based on a rolling 21-day annualised volatility of the portfolio return series.

In [ ]:
r_vol      = compute_realised_vol(portfolio_returns, window=REGIME_VOL_LOOKBACK)
vol_regime = classify_vol_regime(r_vol, REGIME_VOL_LOW, REGIME_VOL_HIGH)

current_vol       = float(r_vol.iloc[-1])
current_vol_label = vol_regime.iloc[-1]

print(f'Current realised vol (ann.) : {current_vol*100:.2f}%')
print(f'Current vol regime          : {current_vol_label}')
print(f'Thresholds                  : low < {REGIME_VOL_LOW*100:.0f}%  |  high > {REGIME_VOL_HIGH*100:.0f}%')

vol_counts = vol_regime.value_counts(normalize=True).round(3)
print(f'\nHistorical regime distribution (full lookback):')
for label, pct in vol_counts.items():
    print(f'   {label:12s}: {pct*100:.1f}%')

## 3. Correlation compression

Second component. When assets all start moving together (average pairwise correlation rises),
it signals stress — diversification is breaking down, even if individual volatilities look normal.

In [ ]:
print('Computing correlation compression (30-day rolling)...')
corr_compression = compute_correlation_compression(log_returns, window=30)

current_compression = float(corr_compression.iloc[-1])
print(f'\nCurrent corr compression : {current_compression:.3f}')
print(f'High-stress threshold    : {REGIME_CORR_COMPRESSION_HIGH}')
print(f'Status                   : {"⚠️  ELEVATED" if current_compression > REGIME_CORR_COMPRESSION_HIGH else "✅ Normal"}')
print(f'\nHistorical percentiles:')
for p in [25, 50, 75, 90]:
    print(f'   P{p:2d}: {np.percentile(corr_compression.dropna(), p):.3f}')

## 4. Composite stress score

Combines vol (60% weight) and correlation compression (40% weight) into a single 0–1 stress score.

- **< 0.35** → `low_stress` (calm market, full deployment)
- **0.35 – 0.65** → `medium` (watch for deterioration)
- **> 0.65** → `high_stress` (turbulent, consider defensive positioning)

In [ ]:
regime_df = compute_composite_regime(
    portfolio_log_returns = portfolio_returns,
    log_returns_all       = log_returns,
    vol_window            = REGIME_VOL_LOOKBACK,
    corr_window           = 30,
    vol_low               = REGIME_VOL_LOW,
    vol_high              = REGIME_VOL_HIGH,
    corr_high             = REGIME_CORR_COMPRESSION_HIGH,
)

current = regime_df.iloc[-1]
print('Current regime snapshot:')
print(f"  Regime          : {current['regime']}")
print(f"  Stress score    : {current['stress_score']:.3f}")
print(f"  Realised vol    : {current['realised_vol']*100:.2f}% ann.")
print(f"  Corr compression: {current['corr_compression']:.3f}")
print(f"  Vol component   : {current['vol_component']:.3f}")
print(f"  Corr component  : {current['corr_component']:.3f}")

print(f'\nFull regime DataFrame (last 10 rows):')
display(regime_df.set_index('date').tail(10).round(4))

## 5. Regime probabilities

In [ ]:
probs = compute_regime_probabilities(regime_df, lookback_days=REGIME_PROB_LOOKBACK)

print(f'Regime probabilities (last {REGIME_PROB_LOOKBACK} trading days):')
for regime, prob in probs.items():
    bar = '█' * int(prob * 40)
    print(f'  {regime:12s}: {prob*100:5.1f}%  {bar}')

## 6. Transition statistics

In [ ]:
transition_stats = compute_transition_stats(regime_df)

print(f"Current regime         : {transition_stats['current_regime']}")
print(f"Days in current regime : {transition_stats['days_in_current_regime']}")
print(f"\nAverage duration per regime (historical):")
for regime, avg_days in transition_stats['avg_duration_per_regime'].items():
    print(f"  {regime:12s}: {avg_days:.1f} days")

## 7. Visualisations

In [ ]:
# ── Plot 1: Stress score over time with regime colouring ──

REGIME_COLORS = {
    'low_stress':  '#4CAF50',
    'medium':      '#FF9800',
    'high_stress': '#F44336',
}

fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)
plot_df = regime_df.set_index('date')

# Panel 1: Stress score
ax = axes[0]
ax.plot(plot_df.index, plot_df['stress_score'], linewidth=1.2, color='#333')
ax.axhline(0.35, color='orange', linestyle='--', linewidth=0.8, label='Medium threshold (0.35)')
ax.axhline(0.65, color='red',    linestyle='--', linewidth=0.8, label='High threshold (0.65)')
ax.fill_between(plot_df.index, plot_df['stress_score'],
                where=plot_df['regime'] == 'high_stress',
                color='red', alpha=0.15, label='High stress')
ax.fill_between(plot_df.index, plot_df['stress_score'],
                where=plot_df['regime'] == 'medium',
                color='orange', alpha=0.12, label='Medium')
ax.fill_between(plot_df.index, plot_df['stress_score'],
                where=plot_df['regime'] == 'low_stress',
                color='green', alpha=0.10, label='Low stress')
ax.set_ylabel('Stress score')
ax.set_ylim(0, 1)
ax.set_title('Composite stress score over time')
ax.legend(fontsize=9, loc='upper left')

# Panel 2: Realised volatility
ax = axes[1]
ax.plot(plot_df.index, plot_df['realised_vol'] * 100, linewidth=1.2, color='steelblue')
ax.axhline(REGIME_VOL_LOW  * 100, color='green',  linestyle='--', linewidth=0.8)
ax.axhline(REGIME_VOL_HIGH * 100, color='red',    linestyle='--', linewidth=0.8)
ax.set_ylabel('Ann. volatility (%)')
ax.set_title('Realised portfolio volatility (21-day rolling)')

# Panel 3: Correlation compression
ax = axes[2]
ax.plot(plot_df.index, plot_df['corr_compression'], linewidth=1.2, color='coral')
ax.axhline(REGIME_CORR_COMPRESSION_HIGH, color='red', linestyle='--', linewidth=0.8,
           label=f'High-stress threshold ({REGIME_CORR_COMPRESSION_HIGH})')
ax.set_ylabel('Avg pairwise |corr|')
ax.set_title('Cross-asset correlation compression (30-day rolling)')
ax.set_xlabel('Date')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Regime probability bar chart (current snapshot) ──

fig, ax = plt.subplots(figsize=(8, 3))

labels = list(probs.keys())
values = [probs[k] * 100 for k in labels]
colors = [REGIME_COLORS[k] for k in labels]

bars = ax.barh(labels, values, color=colors, edgecolor='white', height=0.55)
for bar, val in zip(bars, values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=11)

ax.set_xlim(0, 110)
ax.set_xlabel('Probability (%)')
ax.set_title(f'Regime probabilities — last {REGIME_PROB_LOOKBACK} trading days')
plt.tight_layout()
plt.show()

## 8. Comparison: new model vs existing 200-day SMA filter

Shows how the new probabilistic regime model compares to the binary SMA filter
already in `rules_engine.py`. The SMA filter only acts on individual asset prices;
the new model acts on portfolio-level dynamics.

In [ ]:
# Reconstruct the 200-day SMA benchmark signal
from src.config import TREND_FILTER_MA_PERIODS

benchmark_price = prices[BENCHMARK_TICKER].dropna()
sma_200         = benchmark_price.rolling(TREND_FILTER_MA_PERIODS).mean()
sma_signal      = (benchmark_price > sma_200).astype(int)  # 1 = above SMA (invested), 0 = below (cash)

# Map our regime to binary: low/medium = invested (1), high_stress = cash (0)
regime_signal = plot_df['regime'].map({
    'low_stress':  1,
    'medium':      1,
    'high_stress': 0,
})

# Align both signals
comparison = pd.DataFrame({
    'sma_200_signal':    sma_signal,
    'regime_signal':     regime_signal,
    'benchmark_price':   benchmark_price,
}).dropna()

# Where do they disagree?
comparison['disagree'] = (comparison['sma_200_signal'] != comparison['regime_signal']).astype(int)
disagree_pct = comparison['disagree'].mean() * 100

print(f'Signal disagreement rate : {disagree_pct:.1f}% of trading days')
print(f'Regime says CASH when SMA says INVEST : {((comparison["regime_signal"]==0) & (comparison["sma_200_signal"]==1)).sum()} days')
print(f'SMA says CASH when Regime says INVEST : {((comparison["regime_signal"]==1) & (comparison["sma_200_signal"]==0)).sum()} days')

# Plot comparison
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(comparison.index, comparison['benchmark_price'], linewidth=1, color='#888', label='Benchmark price', zorder=1)
ax.fill_between(comparison.index, comparison['benchmark_price'].min(), comparison['benchmark_price'].max(),
                where=comparison['sma_200_signal'] == 0,
                color='orange', alpha=0.2, label='SMA says CASH')
ax.fill_between(comparison.index, comparison['benchmark_price'].min(), comparison['benchmark_price'].max(),
                where=comparison['regime_signal'] == 0,
                color='red', alpha=0.2, label='Regime says CASH')
ax.set_title('200-day SMA filter vs composite regime model — CASH signals')
ax.set_ylabel('Benchmark price (€)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 9. Export to JSON

In [ ]:
# Prepare regime history (last 252 days) for the dashboard chart
history_cols   = ['date', 'realised_vol', 'corr_compression', 'stress_score', 'regime']
regime_history = (
    regime_df[history_cols]
    .tail(252)
    .assign(date=lambda df: df['date'].astype(str))
    .round(4)
    .to_dict(orient='records')
)

state = {
    'generated_at':        datetime.now().isoformat(),
    'data_end_date':       str(log_returns.index[-1].date()),

    # Current snapshot
    'current_regime':      current['regime'],
    'stress_score':        round(float(current['stress_score']), 4),
    'realised_vol_ann':    round(float(current['realised_vol']), 4),
    'corr_compression':    round(float(current['corr_compression']), 4),
    'vol_component':       round(float(current['vol_component']), 4),
    'corr_component':      round(float(current['corr_component']), 4),

    # Regime probabilities (last REGIME_PROB_LOOKBACK days)
    'regime_probabilities': probs,

    # Transition stats
    'transition_stats':    transition_stats,

    # Thresholds used (so dashboard can display them)
    'thresholds': {
        'vol_low':           REGIME_VOL_LOW,
        'vol_high':          REGIME_VOL_HIGH,
        'corr_high':         REGIME_CORR_COMPRESSION_HIGH,
        'stress_medium':     0.35,
        'stress_high':       0.65,
    },

    # Historical time series for charts (last 252 trading days)
    'regime_history': regime_history,
}

os.makedirs(os.path.dirname(OUTPUT_REGIME), exist_ok=True)
with open(OUTPUT_REGIME, 'w', encoding='utf-8') as f:
    json.dump(state, f, indent=2, default=str)

print(f'✅ Exported to: {OUTPUT_REGIME}')
print(f'   File size : {os.path.getsize(OUTPUT_REGIME) / 1024:.1f} KB')
print(f'\nCurrent regime: {state["current_regime"].upper()}  |  Stress score: {state["stress_score"]}')

---
## Done

`regime_state.json` is now ready and will appear in the dashboard Research tab.

Next steps:
- Run `03_factor_model.ipynb` to decompose returns into market / size / value / alpha
- The regime probabilities from this notebook will feed into `04_black_litterman.ipynb` as confidence weights on views